<a href="https://colab.research.google.com/github/arushikumarics24-beep/AI-LAB/blob/main/1WN24CS051_Week7_FOL_Unification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import re

# Check whether a term is a variable
def is_variable(term):
    return isinstance(term, str) and term[0].islower()


# Parse an expression into a tree structure
def parse(expression):
    tokens = re.findall(r'[A-Za-z_]\w*|[(),]', expression)
    position = [0]

    def build():
        name = tokens[position[0]]
        position[0] += 1

        if position[0] < len(tokens) and tokens[position[0]] == '(':
            position[0] += 1
            args = []

            while True:
                args.append(build())

                if tokens[position[0]] == ')':
                    position[0] += 1
                    break

                position[0] += 1  # Skip comma

            return (name, args)

        return name

    return build()


# Apply existing substitutions
def apply_substitution(term, subst):
    if is_variable(term):
        if term in subst:
            return apply_substitution(subst[term], subst)
        return term

    if isinstance(term, tuple):
        return (
            term[0],
            [apply_substitution(arg, subst) for arg in term[1]]
        )

    return term


# Occurs check
def occurs(variable, term, subst):
    term = apply_substitution(term, subst)

    if variable == term:
        return True

    if isinstance(term, tuple):
        return any(occurs(variable, arg, subst)
                   for arg in term[1])

    return False


# Main Unification Algorithm
def unify(x, y, subst=None):

    if subst is None:
        subst = {}

    x = apply_substitution(x, subst)
    y = apply_substitution(y, subst)

    # Step 1: If both expressions are identical
    if x == y:
        return subst

    # Step 2: If x is a variable
    if is_variable(x):
        if occurs(x, y, subst):
            return None

        subst[x] = y
        return subst

    # Step 3: If y is a variable
    if is_variable(y):
        if occurs(y, x, subst):
            return None

        subst[y] = x
        return subst

    # Step 4: Both must be compound expressions
    if isinstance(x, tuple) and isinstance(y, tuple):

        # Function names must match
        if x[0] != y[0]:
            return None

        # Number of arguments must match
        if len(x[1]) != len(y[1]):
            return None

        # Unify corresponding arguments
        for a, b in zip(x[1], y[1]):
            subst = unify(a, b, subst)

            if subst is None:
                return None

        return subst

    # Different constants or incompatible expressions
    return None


# Convert expression tree back into readable form
def display(term):
    if isinstance(term, tuple):
        return term[0] + "(" + ", ".join(
            display(arg) for arg in term[1]
        ) + ")"

    return term


# Main program
expr1 = input("Enter first expression: ")
expr2 = input("Enter second expression: ")

term1 = parse(expr1)
term2 = parse(expr2)

print("\nExpression 1:", display(term1))
print("Expression 2:", display(term2))

result = unify(term1, term2)

if result is None:
    print("\nUnification Failed!")
else:
    print("\nUnification Successful!")
    print("Substitution:", result)

    unified = apply_substitution(term1, result)

    print("Unified Expression:", display(unified))


Enter first expression: eats(x,apple)
Enter second expression: eats(riya,y)

Expression 1: eats(x, apple)
Expression 2: eats(riya, y)

Unification Successful!
Substitution: {'x': 'riya', 'apple': 'y'}
Unified Expression: eats(riya, y)
